In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer # Can be adapted for k-mers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import itertools
import os # For checking file existence

In [2]:
DATA_FILE = 'dataset.csv'
SEPARATOR = ','

In [3]:
SEQUENCE_COLUMN = 'sequence' # <--- *** Modify here: Replace with the column name containing sequences ***
LABEL_COLUMN = 'feature'     # <--- *** Modify here: Replace with the column name containing feature labels ***
KMER_K = 5                  # <--- Length of K-mer (can be adjusted, 3-7 is common range)
TEST_SET_SIZE = 0.2         # --- Ratio of the test set (e.g., 0.2 for 20%)
RANDOM_SEED = 42            # --- Used to ensure reproducibility

In [4]:
def get_kmers(sequence, k):
    """Generate all k-mers of length k from a sequence"""
    if pd.isna(sequence) or len(sequence) < k:
        return []
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

# *** NEW: Define the analyzer as a regular function ***
def kmer_analyzer(sequence):
    """
    Analyzer function for CountVectorizer.
    It calls get_kmers and uses the KMER_K defined at the top level.
    """
    # Directly use the global KMER_K defined in the Configuration section above
    return get_kmers(sequence, KMER_K)

def build_kmer_vocabulary(k):
    """Generate all possible DNA k-mers (A, C, G, T)"""
    bases = ['A', 'C', 'G', 'T']
    # Use itertools.product to generate all combinations of length k
    # e.g., k=2 -> ('A', 'A'), ('A', 'C'), ..., ('T', 'T')
    # Then use ''.join to convert tuple ('A', 'A') to string "AA"
    kmer_iter = itertools.product(bases, repeat=k)
    return [''.join(kmer) for kmer in kmer_iter]

In [5]:
df = pd.read_csv(DATA_FILE, sep=SEPARATOR)
print(f"Data loaded successfully, total {len(df)} records.")
print("Preview of first 5 rows:")
print(df.head())
print("\nColumn names:", df.columns.tolist())

In [6]:
sequences = df[SEQUENCE_COLUMN].astype(str) # Ensure string type
labels = df[LABEL_COLUMN]
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
print("Label Encoding:")
class_mapping = {label: index for index, label in enumerate(label_encoder.classes_)}
print(f"Class Mapping: {class_mapping}")
print(f"Encoded Label Examples (First 10): {y[:10]}")

In [7]:
kmer_vocab = build_kmer_vocabulary(KMER_K)
print(f"Generated {len(kmer_vocab)} possible k-mers (Vocabulary size)")

In [8]:
# Using the CountVectorizer concept acting on our custom k-mer extraction function
# analyzer=kmer_analyzer specifies how to extract "words" (k-mers) from a "document" (sequence)
# vocabulary=kmer_vocab ensures all sequences use the same feature set and consistent order

# *** CHANGE HERE: Use the named function 'kmer_analyzer' ***
vectorizer = CountVectorizer(analyzer=kmer_analyzer,  # <--- Using named function
                             vocabulary=kmer_vocab)

# Convert all sequences to k-mer count vector matrices
# fit_transform learns the vocabulary (if not provided) and transforms data
# transform only uses existing vocabulary to transform data
# Since we provided vocabulary, transform or fit_transform have the same effect, but transform is faster
print("Converting sequences to k-mer count vectors...")
X = vectorizer.transform(sequences)
print("Conversion complete.")
print(f"Feature matrix X shape: {X.shape}") # (Samples, Features=k-mer vocabulary size)

In [9]:
# 4. Dataset Splitting
print(f"\n--- 4. Dataset Splitting ---")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SET_SIZE,
    random_state=RANDOM_SEED,
    stratify=y  # Ensures similar class proportions in train and test sets
)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

In [10]:
# 5. Model Selection and Training
print(f"\n--- 5. Model Training ---")

# Model 1: Support Vector Machine (SVM)
# print("Training SVM model...")
# svm_model = SVC(kernel='linear', C=1.0, random_state=RANDOM_SEED, probability=True)
# svm_model.fit(X_train, y_train)
# print("SVM model training complete.")

# Model 2: Random Forest
print("Training Random Forest model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1) # n_jobs=-1 uses all available CPU cores
rf_model.fit(X_train, y_train)
print("Random Forest model training complete.")


In [11]:
# 6. Model Evaluation
print(f"\n--- 6. Model Evaluation (on Test Set) ---")

# models = {'SVM': svm_model, 'Random Forest': rf_model}
models = {'Random Forest': rf_model}
for name, model in models.items():
    print(f"\nEvaluating Model: {name}")
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)
    print("-" * 30)

In [12]:
# Select a model for future predictions (e.g., choosing the better performer, here assuming Random Forest)
final_model = rf_model # <--- *** Can select SVM or RF based on results ***
print(f"\nSelected {type(final_model).__name__} as the final model for prediction.")

In [21]:
import joblib
import os

print(f"\n--- 8. Saving Model and Preprocessors ---")

# Define paths and names for saving files
output_dir = "saved_model"
os.makedirs(output_dir, exist_ok=True) # Create save directory if it doesn't exist

MODEL_FILENAME = os.path.join(output_dir, "bacteria_classifier_model.joblib")
VECTORIZER_FILENAME = os.path.join(output_dir, "kmer_vectorizer.joblib")
ENCODER_FILENAME = os.path.join(output_dir, "label_encoder.joblib")
# (Optional) Save K value; although implied in vectorizer, explicit saving is safer
KMER_VALUE_FILENAME = os.path.join(output_dir, "kmer_k_value.joblib")


# Save Model
joblib.dump(final_model, MODEL_FILENAME)
print(f"Model saved to: {MODEL_FILENAME}")

# Save Vectorizer
joblib.dump(vectorizer, VECTORIZER_FILENAME)
print(f"Vectorizer saved to: {VECTORIZER_FILENAME}")

# Save LabelEncoder
joblib.dump(label_encoder, ENCODER_FILENAME)
print(f"LabelEncoder saved to: {ENCODER_FILENAME}")

# Save K Value
joblib.dump(KMER_K, KMER_VALUE_FILENAME)
print(f"K value ({KMER_K}) saved to: {KMER_VALUE_FILENAME}")
